# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [22]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_ROOT = Path("../../data/raw")
assert DATA_ROOT.exists(), (
    "data/raw not found - this repo does not ship raw data (public-data rule). "
    "Download the files locally from the dataset's Hugging Face 'Files and versions' tab "
    "and place them under data/raw/ before running this notebook."
)

DEV_FILE = str(DATA_ROOT / "fact_content_daily_performance" / "data_14.parquet")
DECISION_DATE = "2026-03-15"

con = duckdb.connect()

window_bounds = con.execute(f"SELECT MIN(report_date) AS w_start FROM read_parquet('{DEV_FILE}')").df()
WINDOW_START = str(window_bounds['w_start'][0].date())
print("past window:", WINDOW_START, "->", DECISION_DATE)

# Client-level filter + censoring flags, computed purely from dates - never from row
# existence in the fact table, since row existence can be sparse for reasons unrelated
# to tracking status.
clients = con.execute(f"""
    SELECT
        client_hash_id, access_profile, is_active, gsc_data_start, ga4_data_start,
        (ga4_data_start IS NOT NULL) AS has_ga4_access,
        (access_profile != 'ga4_only') AS has_gsc_access,

        -- Explicit, three-way-aware flag: a client is GSC-unusable in this window if it
        -- structurally never has GSC (ga4_only), if its start date is unrecorded (metadata
        -- inconsistency), or if tracking starts after this window. Written explicitly instead
        -- of relying on SQL's NULL propagation through ">=", since that behavior is easy to
        -- miss and previously caused an undercount.
        (
            access_profile = 'ga4_only'
            OR gsc_data_start IS NULL
            OR gsc_data_start >= DATE '{DECISION_DATE}'
        ) AS gsc_fully_censored,

        (ga4_data_start IS NOT NULL AND ga4_data_start >= DATE '{DECISION_DATE}') AS ga4_temporally_fully_censored
    FROM read_parquet('{DATA_ROOT / "dim_clients.parquet"}')
    WHERE is_active IS TRUE AND access_profile != 'no_search_or_analytics_access'
""").df()
print("usable clients:", len(clients))

# Content-level filter: drop deleted or unpublished content.
content_meta = con.execute(f"""
    SELECT client_hash_id, content_hash_id, is_deleted, is_published, content_created_date
    FROM read_parquet('{DATA_ROOT / "dim_content.parquet"}')
    WHERE is_deleted IS FALSE AND is_published IS TRUE
""").df()
print("usable content rows:", len(content_meta))

# Remove true cold-start content: never observed in any of the 18 months, in any file.
all_seen_content = con.execute(f"""
    SELECT DISTINCT content_hash_id
    FROM read_parquet('{DATA_ROOT / "fact_content_daily_performance/data_*.parquet"}')
""").df()
seen_ids = set(all_seen_content['content_hash_id'])
content_meta = content_meta[content_meta['content_hash_id'].isin(seen_ids)].copy()
print("content_meta after removing true cold-start:", len(content_meta))

# Past-window feature aggregation, per (client, content) pair.
# GSC block: COALESCE(..., 0) is safe - GSC censoring is negligible, and a zero-row window
# for an otherwise-tracked client genuinely means "no traffic this window."
# GA4 block: NULL is preserved (not zero-filled) whenever a client is censored, because
# GA4 censoring is large (~40%). The outer CASE checks the client's tracking status
# before deciding whether a zero-row result means "real zero" or "not measured."
# ANY_VALUE() is required because ga4_data_start is constant per client but referenced
# outside an aggregate function; this is safe since it never varies within a group.
past_features = con.execute(f"""
    SELECT
        cm.client_hash_id,
        cm.content_hash_id,
        c.has_ga4_access,

        COALESCE(SUM(CASE WHEN f.report_date >= c.gsc_data_start THEN f.gsc_clicks END), 0)      AS clicks_past,
        COALESCE(SUM(CASE WHEN f.report_date >= c.gsc_data_start THEN f.gsc_impressions END), 0)  AS impressions_past,
        AVG(CASE WHEN f.report_date >= c.gsc_data_start THEN f.gsc_avg_position END)               AS avg_position_past,
        COUNT(*) FILTER (WHERE f.report_date >= c.gsc_data_start) AS gsc_days_covered_past,

        CASE
            WHEN c.has_ga4_access AND ANY_VALUE(c.ga4_data_start) < DATE '{DECISION_DATE}'
            THEN COALESCE(SUM(CASE WHEN f.report_date >= c.ga4_data_start THEN f.ga4_engaged_sessions END), 0)
            ELSE NULL
        END AS engaged_sessions_past,

        CASE
            WHEN c.has_ga4_access AND ANY_VALUE(c.ga4_data_start) < DATE '{DECISION_DATE}'
            THEN COALESCE(SUM(CASE WHEN f.report_date >= c.ga4_data_start THEN f.sessions_organic END), 0)
            ELSE NULL
        END AS organic_sessions_past,

        COUNT(*) FILTER (WHERE f.report_date >= c.ga4_data_start) AS ga4_days_covered_past

    FROM content_meta cm
    JOIN clients c
      ON cm.client_hash_id = c.client_hash_id
    LEFT JOIN read_parquet('{DEV_FILE}') f
      ON f.client_hash_id = cm.client_hash_id
     AND f.content_hash_id = cm.content_hash_id
     AND f.report_date < DATE '{DECISION_DATE}'
    GROUP BY 1, 2, 3
""").df()
print("rows before any censoring cleanup:", len(past_features))

# Attach client-level censoring flags.
X = past_features.merge(
    clients[['client_hash_id', 'gsc_fully_censored', 'ga4_temporally_fully_censored']],
    on='client_hash_id', how='left'
)

# Two usable clients have a NULL gsc_data_start despite claiming GSC access - a metadata
# inconsistency (see w03_data_contract.ipynb, Section 4). gsc_fully_censored is NaN for
# their rows because "NULL >= date" evaluates to NULL in SQL, not True/False. These are
# treated conservatively as censored: if the tracking start is unknown, a zero cannot be
# trusted as real. fillna(True) makes this decision explicit instead of letting NaN
# silently propagate through the boolean mask.

fully_censored_gsc = X['gsc_fully_censored']
structural_no_ga4 = ~X['has_ga4_access']
temporal_censored_ga4 = X['has_ga4_access'] & X['ga4_temporally_fully_censored']

print(f"GSC fully censored or unknown start (dropped): {fully_censored_gsc.sum()} ({fully_censored_gsc.mean():.1%})")
print(f"GA4 structurally absent: {structural_no_ga4.sum()} ({structural_no_ga4.mean():.1%})")
print(f"GA4 temporally censored (date-based): {temporal_censored_ga4.sum()} ({temporal_censored_ga4.mean():.1%})")

# Drop rows for clients with no confirmed GSC history in this window. Note: dropping
# these rows also lowers the overall GA4-NaN rate (from ~45% to ~40%), because these
# same clients are largely censored for GA4 too - both metric families reflect the
# same underlying onboarding delay, not two independent missingness patterns.
X = X[~fully_censored_gsc].copy()

print("\nGSC censoring status among the 56 usable clients:")
print(clients['gsc_fully_censored'].value_counts())
print("\ngsc_data_start for the fully-censored clients (all postdate this dev window):")
print(clients[clients['gsc_fully_censored']]['gsc_data_start'].sort_values().to_string())

# Final feature vector: shape, dtypes, and a preview.
FEATURE_COLS = [
    "clicks_past", "impressions_past", "avg_position_past",
    "engaged_sessions_past", "organic_sessions_past",
]

print("\nengaged_sessions_past NaN count:", X['engaged_sessions_past'].isna().sum(),
      f"({X['engaged_sessions_past'].isna().mean():.1%})")

print("\nFeature vector shape:", X[FEATURE_COLS].shape)
print("Data types:")
print(X[FEATURE_COLS].dtypes)
X[FEATURE_COLS].head(30)

past window: 2026-03-01 -> 2026-03-15
usable clients: 56
usable content rows: 411540
content_meta after removing true cold-start: 402064
rows before any censoring cleanup: 317067
GSC fully censored or unknown start (dropped): 27187 (8.6%)
GA4 structurally absent: 85823 (27.1%)
GA4 temporally censored (date-based): 57037 (18.0%)

GSC censoring status among the 56 usable clients:
gsc_fully_censored
False    38
True     18
Name: count, dtype: int64

gsc_data_start for the fully-censored clients (all postdate this dev window):
27   2026-03-25
50   2026-03-27
31   2026-04-07
1    2026-04-10
34   2026-04-29
44   2026-04-30
6    2026-05-11
45   2026-05-14
24   2026-05-18
37   2026-05-24
40   2026-06-02
0           NaT
15          NaT
26          NaT
32          NaT
48          NaT
53          NaT
54          NaT

engaged_sessions_past NaN count: 115673 (39.9%)

Feature vector shape: (289880, 5)
Data types:
clicks_past              float64
impressions_past         float64
avg_position_past    

,clicks_past,impressions_past,avg_position_past,engaged_sessions_past,organic_sessions_past
0,3.0,848.0,10.280239,0.0,2.0
1,1.0,1265.0,20.635815,0.0,0.0
2,0.0,396.0,10.910397,0.0,0.0
3,0.0,518.0,7.725086,0.0,0.0
4,0.0,363.0,7.711800,0.0,0.0
5,2.0,661.0,32.435331,0.0,1.0
6,18.0,4873.0,22.254636,2.0,14.0
7,0.0,626.0,44.960353,0.0,0.0
8,0.0,250.0,18.882896,0.0,0.0
9,0.0,1693.0,32.646561,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [14]:
feature_notes = {
    "clicks_past": "available at decision date - summed only over days >= this client's gsc_data_start",
    "impressions_past": "available at decision date - same rule as clicks_past",
    "avg_position_past": "available at decision date - same rule as clicks_past",
    "gsc_days_covered_past": "coverage counter, not a growth signal - number of past-window days actually measurable for GSC",
    "engaged_sessions_past": (
        "available at decision date for clients with GA4 access - summed only over days >= "
        "ga4_data_start; NaN if structurally no GA4 access, or if the whole past window predates "
        "this client's GA4 tracking start"
    ),
    "organic_sessions_past": "available at decision date - same rule as engaged_sessions_past",
    "ga4_days_covered_past": "coverage counter for GA4, same purpose as gsc_days_covered_past",
    "has_ga4_access": "categorical/boolean context flag - not a growth signal, used to interpret GA4 NaNs correctly",
}
for k, v in feature_notes.items():
    print(f"{k}: {v}")

clicks_past: available at decision date - summed only over days >= this client's gsc_data_start
impressions_past: available at decision date - same rule as clicks_past
avg_position_past: available at decision date - same rule as clicks_past
gsc_days_covered_past: coverage counter, not a growth signal - number of past-window days actually measurable for GSC
engaged_sessions_past: available at decision date for clients with GA4 access - summed only over days >= ga4_data_start; NaN if structurally no GA4 access, or if the whole past window predates this client's GA4 tracking start
organic_sessions_past: available at decision date - same rule as engaged_sessions_past
ga4_days_covered_past: coverage counter for GA4, same purpose as gsc_days_covered_past
has_ga4_access: categorical/boolean context flag - not a growth signal, used to interpret GA4 NaNs correctly


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.impute import SimpleImputer

# Future window: same aggregation style as the past window, used only to build the label,
# never joined into the feature matrix except for the deliberate-leak demo below.
future_outcome = con.execute(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_clicks) AS clicks_future
    FROM read_parquet('{DEV_FILE}') f
    JOIN clients c ON f.client_hash_id = c.client_hash_id
    JOIN content_meta cm ON f.client_hash_id = cm.client_hash_id AND f.content_hash_id = cm.content_hash_id
    WHERE f.report_date >= DATE '{DECISION_DATE}'
    GROUP BY 1, 2
""").df()

X_labeled = X.merge(future_outcome, on=["client_hash_id", "content_hash_id"], how="inner")
# Simple binary proxy label for this quick demo: did clicks go up in the future window?
# (the full 4-class growing/declining/recovering/stable label is defined in the capstone modeling notebook)
X_labeled["label"] = (X_labeled["clicks_future"] > X_labeled["clicks_past"]).astype(int)
print("labeled rows:", len(X_labeled), "| positive rate:", X_labeled["label"].mean().round(3))


feature_cols_clean = [
    "clicks_past", "impressions_past", "avg_position_past",
    "engaged_sessions_past", "organic_sessions_past",
]

def quick_score(df, cols):
    """Fits a quick logistic regression on a time-aware-agnostic split, just to demonstrate the
    leakage effect - NOT the real capstone validation design (that split must be time-aware/grouped,
    built separately in the modeling notebook)."""
    imputer = SimpleImputer(strategy="median")  # quick-demo only; real imputation choice is a modeling-week decision
    X_imp = pd.DataFrame(imputer.fit_transform(df[cols]), columns=cols)
    X_train, X_test, y_train, y_test = train_test_split(
        X_imp, df["label"], test_size=0.3, random_state=42, stratify=df["label"]
    )
    model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    return f1_score(y_test, model.predict(X_test), average="macro")

# 1) DELIBERATE LEAK: inject a column derived directly from the future window / the label itself.
# This is not a real feature - it exists only to show what a leaking model looks like.
X_leaky = X_labeled.copy()
X_leaky["clicks_future_leaked"] = X_leaky["clicks_future"]

leaky_score = quick_score(X_leaky, feature_cols_clean + ["clicks_future_leaked"])
print("LEAKY macro-F1 (should look unrealistically high):", round(leaky_score, 3))

# 2) Remove the leaked column and keep the honest number.
honest_score = quick_score(X_labeled, feature_cols_clean)
print("HONEST macro-F1 (past-window features only):", round(honest_score, 3))
print("score inflation caused by the leak:", round(leaky_score - honest_score, 3))

labeled rows: 254408 | positive rate: 0.125
LEAKY macro-F1 (should look unrealistically high): 1.0
HONEST macro-F1 (past-window features only): 0.519
score inflation caused by the leak: 0.481


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* **AI Referral Metrics (`sessions_ai`, `ai_chatgpt`, `ai_perplexity`, etc.):** Highly sparse across individual (client, content) pairs; kept for exploratory data analysis (EDA) only.
* **Content Generation Metadata (`provider_used`, `model_used`):** Describes the upstream creation mechanism rather than time-varying performance trajectory.
* **Static Content-Shape Features (`char_count`, `word_count`, `keyword_*`, `url_char_count`, `category_count`):** Belong to the content-quality and archetype classification lane, not momentum prediction.
* **Optimization Dates (`last_optimized_date`, `optimization_eligible_date`):** Belong to the target definition of the refresh-scoring lane.
* **Detailed Query Data (`fact_content_query_90d.parquet`):** Formatted at a different grain (client, content, query, 90d) than our primary unit of analysis (client, content, day).
* **Unmeasured Clients (`access_profile == 'no_search_or_analytics_access'`):** Excluded at the client filter stage due to a total lack of tracking telemetry.
* **Inactive/Unpublished Content (`is_deleted == True` or `is_published == False`):** Predictions on dead or draft pages yield no downstream operational value.
* **True Cold-Start Items (~9,476 items):** Content never observed in any historical performance file lacks a reference baseline to define growth or decline; dropped prior to feature calculation.
* **Future/Leaked Signals (`clicks_future_leaked`):** Injected solely for the Section 3 leakage verification demo; omitted from production modeling.


In [ ]:
exclusion_reasons = {
    "sessions_ai (+ per-model AI referral columns)": (
        "too sparse per (client, content) to support a class label reliably; kept for EDA only"
    ),
    "provider_used, model_used": "describes how the content was generated, not how it performs over time",
    "char_count, word_count, keyword_char_count, keyword_token_count, url_char_count, category_count": (
        "static content-shape metadata; belongs to a content-archetype/quality lane, not momentum prediction"
    ),
    "last_optimized_date, optimization_eligible_date": "belongs to the refresh-scoring lane, not this lane's target",
    "fact_content_query_90d.parquet": (
        "different grain (client, content, query, 90d) than this lane's (client, content, day) grain; "
        "mixing grains would corrupt the unit of analysis"
    ),
    "clients with access_profile == 'no_search_or_analytics_access'": (
        "no measurable signal exists for these clients at all; excluded at the client-filter step, not zero-filled"
    ),
    "content with is_deleted == True or is_published == False": (
        "predicting momentum for dead or unpublished pages has no downstream action"
    ),
    "content never observed in any of the 18 months (~9,476 items)": (
        "no baseline traffic in any window - growth/decline cannot be defined without a reference point; "
        "excluded before feature computation, not zero-filled"
    ),
    "content with zero rows in the dev month but observed elsewhere": (
        "kept via LEFT JOIN + zero-fill on click/impression/session sums; "
        "this is a real 'no traffic this window' signal, not missingness"
    ),
    "rows for clients with no confirmed GSC start (27,187 rows, 8.6%)": (
        "25,778 rows from 11 clients whose gsc_data_start postdates this dev window "
        "(confirmed ongoing onboarding, ranging 2026-03-25 to 2026-06-02), plus 1,409 rows "
        "from 2 clients with a NULL gsc_data_start despite claiming GSC access (a metadata "
        "inconsistency, treated conservatively as censored)"
    ),
    "rows structurally or temporally censored for GA4": (
        "kept but left as NaN, split into 'structural' (client will never have GA4) and 'temporal' "
        "(GA4 tracking starts after this past window) - treating either as zero would falsely teach "
        "the model 'no GA4 traffic' when the truth is 'not measured'"
    ),
    "clicks_future_leaked": (
        "deliberately-injected leak used only to demonstrate the trap in Section 3; never used in the real model"
    ),
}
for k, v in exclusion_reasons.items():
    print(f"- {k}: {v}")

# Sanity check: confirm the Section 1 cold-start filter left no unmeasurable content behind.
# Expected: True, since content_meta was already filtered against seen_ids in Section 1.
assert content_meta['content_hash_id'].isin(seen_ids).all(), "cold-start content leaked through the filter"
print("\nsanity check passed: no cold-start content remains in content_meta")

- sessions_ai (+ per-model AI referral columns): too sparse per (client, content) to support a class label reliably; kept for EDA only
- provider_used, model_used: describes how the content was generated, not how it performs over time
- char_count, word_count, keyword_char_count, keyword_token_count, url_char_count, category_count: static content-shape metadata; belongs to a content-archetype/quality lane, not momentum prediction
- last_optimized_date, optimization_eligible_date: belongs to the refresh-scoring lane, not this lane's target
- fact_content_query_90d.parquet: different grain (client, content, query, 90d) than this lane's (client, content, day) grain; mixing grains would corrupt the unit of analysis
- clients with access_profile == 'no_search_or_analytics_access': no measurable signal exists for these clients at all; excluded at the client-filter step, not zero-filled
- content with is_deleted == True or is_published == False: predicting momentum for dead or unpublished pag

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.